# RNN Mahcine Translation

# Import libraries

In [24]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, Dense, Input, Dropout, LayerNormalization
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, TimeDistributed, RepeatVector
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.optimizers import Adam

# Load Dataset

In [3]:
en_df = pd.read_csv('/kaggle/input/french-to-english/small_vocab_en.csv', header=None, usecols=[0])
fr_df =  pd.read_csv('/kaggle/input/french-to-english/small_vocab_fr.csv',header=None, usecols=[0]) 

In [5]:
english_sentences =  en_df[0].values
french_sentences = fr_df[0].values
print(english_sentences)
print(french_sentences)
for i in range(len(english_sentences)): 
    english_sentences[i] = "sos " + str(english_sentences[i]) + " eos."
    french_sentences[i] = "sos " + str(french_sentences[i]) + " eos."
english_sentences[0]

['new jersey is sometimes quiet during autumn '
 'the united states is usually chilly during july '
 'california is usually quiet during march ' ...
 'india is never wet during summer '
 'france is never chilly during january '
 'the orange is her favorite fruit ']
["new jersey est parfois calme pendant l' automne "
 'les états-unis est généralement froid en juillet '
 'california est généralement calme en mars ' ...
 "l' inde est jamais mouillé pendant l' été "
 'la france est jamais froid en janvier '
 "l'orange est son fruit préféré "]


'sos new jersey is sometimes quiet during autumn  eos.'

# Preprocessing and Tokenization

In [6]:
#Preprocessing Special characters 
num_words = 10000
tokenizer_en = Tokenizer(num_words=num_words, filters='!#$%&()*+,-/:;<=>@«»""[\\]^_`{|}~\t\n')
tokenizer_en.fit_on_texts(english_sentences)
english_sentences = tokenizer_en.texts_to_sequences(english_sentences)

word_index = tokenizer_en.word_index
print(f"The number of words in the English vocabulary: {len(word_index)}")

The number of words in the English vocabulary: 217


In [7]:
# Embeddings = text to sequences 
#A dictionary (word_index) is created where each unique word in the text is assigned a unique integer.
tokenizer_fr = Tokenizer(num_words=num_words, filters='!#$%&()*+,-/:;<=>@«»""[\\]^_`{|}~\t\n')
tokenizer_fr.fit_on_texts(french_sentences)
french_sentences = tokenizer_fr.texts_to_sequences(french_sentences)

word_index_fr = tokenizer_fr.word_index
print(f"The number of words in the French vocabulary: {len(word_index_fr)}")

The number of words in the French vocabulary: 339


In [8]:
english_sentences = pad_sequences(english_sentences, maxlen = 30, padding='post', truncating='post')
french_sentences = pad_sequences(french_sentences, maxlen=30, padding='post', truncating='post')
print(english_sentences)
print(len(english_sentences))

[[ 1 13 19 ...  0  0  0]
 [ 1  5 16 ...  0  0  0]
 [ 1 18  3 ...  0  0  0]
 ...
 [ 1 15  3 ...  0  0  0]
 [ 1 20  3 ...  0  0  0]
 [ 1  5 78 ...  0  0  0]]
137860


# RNN Model

In [25]:
embedding_dim = 64
latent_dim = 64
num_words = 10000
max_sequence_length = 30
vocab_size_en = len(word_index) + 1  # +1 for padding token
vocab_size_fr = len(word_index_fr) + 1  # +1 for padding token

In [27]:
#Define the encoder
encoder_inputs = Input(shape=(None,))
encoder_embedding = Embedding(vocab_size_en, embedding_dim)(encoder_inputs)
encoder_rnn = SimpleRNN(latent_dim, return_state=True)(encoder_embedding)
encoder_states = encoder_rnn[1:]

# Define the decoder
decoder_inputs = Input(shape=(None,))
decoder_embedding = Embedding(vocab_size_fr, embedding_dim)(decoder_inputs)
decoder_rnn = SimpleRNN(latent_dim, return_sequences=True)(decoder_embedding, initial_state=encoder_states)
decoder_outputs = TimeDistributed(Dense(vocab_size_fr, activation='softmax'))(decoder_rnn)

# Training and Evaluation

In [28]:
# Build the Model
# Model([encoder_inputs, decoder_inputs], decoder_outputs)

model = tf.keras.Model([encoder_inputs, decoder_inputs], decoder_outputs)

# Compile the Model
model.compile(optimizer=Adam(), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Summary of the Model
model.summary()

# Train the model with the padded data
model.fit([english_sentences, decoder_input_data], decoder_target_data, batch_size=64, epochs=10, validation_split=0.2)


Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_8             │ (None, None)           │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ input_layer_9             │ (None, None)           │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ embedding_8 (Embedding)   │ (None, None, 64)       │         13,952 │ input_layer_8[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ embedding_9 (Embedding)   │ (None, None, 64)       │         21,760 │ input_layer_9[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ simple_rnn_2 (SimpleRNN)  │ [(None, 64), (None,    │          8,256 │ embedding_8[0][0]      │
│                           │ 64)]                   │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ simple_rnn_3 (SimpleRNN)  │ (None, None, 64)       │          8,256 │ embedding_9[0][0],     │
│                           │                        │                │ simple_rnn_2[0][1]     │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ time_distributed_4        │ (None, None, 340)      │         22,100 │ simple_rnn_3[0][0]     │
│ (TimeDistributed)         │                        │                │                        │
└───────────────────────────┴────────────────────────┴────────────────┴────────────────────────┘

 Total params: 74,324 (290.33 KB)

 Trainable params: 74,324 (290.33 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
1724/1724 ━━━━━━━━━━━━━━━━━━━━ 65s 27ms/step - accuracy: 0.8546 - loss: 0.8628 - val_accuracy: 0.9335 - val_loss: 0.2077
Epoch 2/10
1724/1724 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - accuracy: 0.9377 - loss: 0.1890 - val_accuracy: 0.9477 - val_loss: 0.1481
Epoch 3/10
1724/1724 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - accuracy: 0.9516 - loss: 0.1376 - val_accuracy: 0.9573 - val_loss: 0.1195
Epoch 4/10
1724/1724 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - accuracy: 0.9594 - loss: 0.1137 - val_accuracy: 0.9625 - val_loss: 0.1043
Epoch 5/10
1724/1724 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - accuracy: 0.9628 - loss: 0.1016 - val_accuracy: 0.9628 - val_loss: 0.1003
Epoch 6/10
1724/1724 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - accuracy: 0.9647 - loss: 0.0949 - val_accuracy: 0.9633 - val_loss: 0.0982
Epoch 7/10
1724/1724 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - accuracy: 0.9655 - loss: 0.0918 - val_accuracy: 0.9661 - val_loss: 0.0903
Epoch 8/10
1724/1724 ━━━━━━━━━━━━━━━━━━━━ 15s 9ms/step - accuracy: 0.9667 - loss: 